# Mixed xLSTM GPU training (Riccio)

**Flow:** (1) resolve repo + imports + `device`, (2) **smoke test** — data, one forward/backward on GPU, (3) **full training** — large model + epochs + checkpoint + test metrics.

Run cells **top to bottom**. Smoke must pass before full training runs.

## 1. Repo root, imports, and device

The notebook searches **upward from the current working directory**, then **shallow-scans `/content`** (Colab). If it still fails: set `REPO_ROOT_MANUAL = "/path/to/repo"` at the top of the code cell (the folder that contains `fitness_coach/`), or run `%cd /path/to/repo` once, or set env `FITNESS_COACH_REPO`.

In [14]:
# Check if running in Colab
import sys
try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("⚠ Not running in Colab - some features may not work as expected")

# Check GPU availability
import torch
print(f"\n GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" GPU Name: {torch.cuda.get_device_name(0)}")
    print(f" GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

✓ Running in Google Colab

 GPU Available: True
 GPU Name: Tesla T4
 GPU Memory: 15.6 GB


In [19]:
import os
import subprocess
from pathlib import Path

# Official capstone repo (must match your GitHub project)
CAPSTONE_REPO_URL = os.environ.get(
    "CAPSTONE_REPO_URL",
    "https://github.com/Elina425/Finess-coach-capstone.git",
)

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    print("✓ Google Drive mounted at /content/drive")

    WORKSPACE_ROOT = Path("/content/drive/My Drive/Finess-coach-capstone")
    if (WORKSPACE_ROOT / ".git").is_dir():
        print("Updating existing clone (git pull)…")
        subprocess.run(["git", "-C", str(WORKSPACE_ROOT), "pull", "--ff-only"], check=False)
    else:
        if WORKSPACE_ROOT.exists() and any(WORKSPACE_ROOT.iterdir()):
            raise RuntimeError(
                f"{WORKSPACE_ROOT} exists but is not a git repo. "
                "Rename/remove that folder on Drive, or set CAPSTONE_REPO_URL and use an empty path."
            )
        WORKSPACE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        print(f"Cloning {CAPSTONE_REPO_URL} → {WORKSPACE_ROOT}")
        subprocess.run(
            ["git", "clone", "--depth", "1", CAPSTONE_REPO_URL, str(WORKSPACE_ROOT)],
            check=True,
        )
else:
    WORKSPACE_ROOT = Path.cwd()

print(f"\nWorkspace root: {WORKSPACE_ROOT}")
print(f"Exists: {WORKSPACE_ROOT.exists()}")
if not (WORKSPACE_ROOT / "fitness_coach").is_dir():
    print("⚠ Expected package dir fitness_coach/ not found — check clone path.")

MessageError: User cancelled dfs_ephemeral authorization

## 2. Paths and hyperparameters

- **Data:** `results/riccio_realtime_exercise_recognition/{stem}_*.npz` (same as `train_xlstm_keypoints.py --feature-mode mixed`).
- **Smoke:** tiny model + one training step + short val pass.
- **Full:** large model; lower `full_batch_size` if you hit OOM on T4.

In [17]:
stem = "riccio_realtime_exercise_recognition"
data_dir = REPO_ROOT / "results" / "riccio_realtime_exercise_recognition"
results_dir = REPO_ROOT / "results" / "xlstm_mixed_large"
results_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = results_dir / "xlstm_keypoints_best.pt"

window = 30
stride = 15

# --- smoke (fast) ---
smoke_hidden = 128
smoke_layers = 2
smoke_batch = 8

# --- full GPU run ---
full_hidden = 512
full_layers = 6
full_batch_size = 16  # reduce to 8 if OOM
full_lr = 4e-4
full_dropout = 0.25
full_epochs = 40

required = [
    data_dir / f"{stem}_biomechanics.npz",
    data_dir / f"{stem}_keypoints.npz",
    data_dir / f"{stem}_labels.npz",
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing Riccio NPZs:\n" + "\n".join(missing))

print("data_dir:", data_dir)
print("results_dir:", results_dir)
print("checkpoint:", ckpt_path)

NameError: name 'REPO_ROOT' is not defined

## 3. Smoke test (imports, data, GPU forward/backward)

Run this cell after sections 1–2. It builds datasets, runs **one** optimizer step on a real batch, and one validation batch. If anything fails, fix paths or install deps before full training.

In [20]:
def run_smoke_test() -> None:
    if not torch.cuda.is_available():
        print("[smoke] WARNING: CUDA not available; smoke still runs on CPU.")

    train_ds, val_ds, test_ds, class_to_idx, idx_to_class, mean, std = build_kaggle_mixed_datasets(
        data_dir,
        stem=stem,
        window=window,
        stride=stride,
        standardize=True,
    )
    n_classes = len(class_to_idx)
    feat_dim = int(train_ds[0][0].shape[-1])
    assert feat_dim == 42, f"expected mixed dim 42, got {feat_dim}"
    print("[smoke] classes:", list(class_to_idx.keys()))
    print("[smoke] train/val/test:", len(train_ds), len(val_ds), len(test_ds))

    train_loader = DataLoader(train_ds, batch_size=smoke_batch, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=smoke_batch, shuffle=False)
    xb, y_cls, y_q = next(iter(train_loader))
    xb = xb.to(device)
    y_cls = y_cls.to(device)
    y_q = y_q.to(device)

    smoke_model = xLSTMExerciseClassifier(
        input_size=feat_dim,
        hidden_size=smoke_hidden,
        num_layers=smoke_layers,
        num_classes=n_classes,
        dropout=0.2,
        bidirectional=True,
    ).to(device)
    smoke_model.train()
    opt = torch.optim.AdamW(smoke_model.parameters(), lr=1e-3, weight_decay=1e-4)
    logits, q_pred = smoke_model(xb)
    q_pred = q_pred.squeeze(-1)
    loss = nn.functional.cross_entropy(logits, y_cls) + 0.5 * nn.functional.mse_loss(q_pred, y_q)
    loss.backward()
    opt.step()
    print("[smoke] train step loss:", float(loss.detach().cpu()))

    smoke_model.eval()
    with torch.no_grad():
        xv, yv, qv = next(iter(val_loader))
        xv = xv.to(device)
        lv, qv_pred = smoke_model(xv)
        pred = lv.argmax(dim=1).cpu()
        acc = float((pred == yv).float().mean())
    print("[smoke] val batch acc (random init):", acc)
    del smoke_model, opt, xb, y_cls, y_q, xv, lv, qv_pred, pred
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("[smoke] OK")


run_smoke_test()

NameError: name 'build_kaggle_mixed_datasets' is not defined

## 4. Build datasets for full training

Reuses the same pipeline as the smoke test (aligned angles + keypoints per window).

In [ ]:
train_ds, val_ds, test_ds, class_to_idx, idx_to_class, mean, std = build_kaggle_mixed_datasets(
    data_dir,
    stem=stem,
    window=window,
    stride=stride,
    standardize=True,
)

input_size = int(train_ds[0][0].shape[-1])
num_classes = len(class_to_idx)
print("input_size:", input_size, "num_classes:", num_classes)
print("Train / val / test:", len(train_ds), len(val_ds), len(test_ds))

## 5. Full training — xLSTM (mixed 42-dim)

Optional: set `resume_from_checkpoint = True` to warm-start from `ckpt_path` if that file already exists (same architecture).

In [ ]:
resume_from_checkpoint = False  # set True to continue from ckpt_path

train_loader = DataLoader(train_ds, batch_size=full_batch_size, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=full_batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=full_batch_size, shuffle=False)

model = xLSTMExerciseClassifier(
    input_size=input_size,
    hidden_size=full_hidden,
    num_layers=full_layers,
    num_classes=num_classes,
    dropout=full_dropout,
    bidirectional=True,
).to(device)

if resume_from_checkpoint and ckpt_path.is_file():
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    model.load_state_dict(sd, strict=True)
    print("Warm-started from", ckpt_path)
else:
    print("Training from scratch")

optimizer = torch.optim.AdamW(model.parameters(), lr=full_lr, weight_decay=1e-4)
criterion_ce = nn.CrossEntropyLoss()
criterion_mse = nn.MSELoss()

best_val_acc = 0.0
best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

for epoch in range(1, full_epochs + 1):
    model.train()
    total_loss = 0.0
    total_samples = 0
    for xb, y_cls, y_q in train_loader:
        xb = xb.to(device)
        y_cls = y_cls.to(device)
        y_q = y_q.to(device)
        optimizer.zero_grad()
        logits, q_pred = model(xb)
        q_pred = q_pred.squeeze(-1)
        loss = criterion_ce(logits, y_cls) + 0.5 * criterion_mse(q_pred, y_q)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        total_loss += float(loss.item()) * xb.size(0)
        total_samples += xb.size(0)
    avg_loss = total_loss / max(total_samples, 1)

    model.eval()
    correct = 0
    total = 0
    val_reg = 0.0
    with torch.no_grad():
        for xb, y_cls, y_q in val_loader:
            xb = xb.to(device)
            y_cls = y_cls.to(device)
            y_q = y_q.to(device)
            logits, q_pred = model(xb)
            q_pred = q_pred.squeeze(-1)
            preds = logits.argmax(dim=1)
            correct += int((preds == y_cls).sum().item())
            total += xb.size(0)
            val_reg += float(criterion_mse(q_pred, y_q).item()) * xb.size(0)
    val_acc = correct / max(total, 1)
    val_rmse = (val_reg / max(total, 1)) ** 0.5
    print(
        f"epoch {epoch:03d} train_loss={avg_loss:.4f} val_acc={val_acc:.4f} val_q_rmse={val_rmse:.4f}"
    )
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print("Best val acc:", best_val_acc)

torch.save(
    {
        "model": best_state,
        "window": window,
        "stride": stride,
        "input_size": input_size,
        "feature_mode": "mixed",
        "bidirectional": True,
        "classes": list(class_to_idx.keys()),
        "class_to_idx": class_to_idx,
        "mean": mean,
        "std": std,
        "hidden": full_hidden,
        "layers": full_layers,
        "dropout": full_dropout,
        "lr": full_lr,
        "batch_size": full_batch_size,
    },
    ckpt_path,
)
print("Saved checkpoint to", ckpt_path)

## 6. Test set evaluation

In [ ]:
def evaluate_model(model, loader, device, class_names):
    model.eval()
    correct = 0
    total = 0
    y_true = []
    y_pred = []
    prob_list = []
    mse = nn.MSELoss()
    reg_loss = 0.0
    abs_err = 0.0
    with torch.no_grad():
        for xb, y_cls, y_q in loader:
            xb = xb.to(device)
            y_cls = y_cls.to(device)
            y_q = y_q.to(device)
            logits, q_pred = model(xb)
            q_pred = q_pred.squeeze(-1)
            preds = logits.argmax(dim=1)
            correct += int((preds == y_cls).sum().item())
            total += xb.size(0)
            reg_loss += float(mse(q_pred, y_q).item()) * xb.size(0)
            abs_err += float(torch.abs(q_pred - y_q).sum().item())
            y_true.extend(y_cls.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())
            prob_list.append(torch.softmax(logits, dim=1).cpu().numpy())
    from sklearn.metrics import (
        accuracy_score,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
    )

    y_true = np.array(y_true, dtype=np.int64)
    y_pred = np.array(y_pred, dtype=np.int64)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted")),
        "precision_macro": float(
            precision_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
        "class_names": class_names,
    }
    reg_metrics = {
        "rmse": float((reg_loss / max(total, 1)) ** 0.5),
        "mae": float(abs_err / max(total, 1)),
    }
    y_prob = np.vstack(prob_list) if prob_list else np.zeros((0, len(class_names)), dtype=np.float32)
    return metrics, reg_metrics, y_true, y_pred, y_prob


metrics, reg_metrics, y_true, y_pred, y_prob = evaluate_model(
    model, test_loader, device, list(class_to_idx.keys())
)
print("Test accuracy:", metrics["accuracy"])
print("F1 weighted:", metrics["f1_weighted"])
print("Quality RMSE:", reg_metrics["rmse"])

## 7. Plots and saved reports

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

x0, y0, q0 = test_ds[0]
x0 = x0.numpy()
angles = x0[:, :8]

plt.figure(figsize=(10, 4))
for i in range(min(4, angles.shape[1])):
    plt.plot(angles[:, i], label=f"angle_{i}")
plt.title("Sample mixed feature angles (first test window)")
plt.xlabel("Frame")
plt.ylabel("Angle value")
plt.legend()
plt.show()

report = classification_report(
    y_true,
    y_pred,
    target_names=list(class_to_idx.keys()),
    zero_division=0,
    output_dict=True,
)
results = {
    "classification_metrics": metrics,
    "regression_metrics": reg_metrics,
    "classification_report": report,
    "num_samples": len(y_true),
}
results_path = results_dir / "mixed_xlstm_evaluation_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print("Saved", results_path)
print(classification_report(y_true, y_pred, target_names=list(class_to_idx.keys()), zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("Test confusion matrix")
plt.colorbar()
ticks = list(class_to_idx.keys())
plt.xticks(range(len(ticks)), ticks, rotation=90)
plt.yticks(range(len(ticks)), ticks)
plt.xlabel("Predicted")
plt.ylabel("True")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="black")
plt.tight_layout()
plt.show()

prob_path = results_dir / "mixed_xlstm_test_probabilities.npy"
np.save(prob_path, y_prob)
print("Saved", prob_path)